# Stage 3 on Kaggle: revision reruns

Checkpoints are selected by validation AUC. Run this notebook four times:

1. `MODE = "lr"`: learning-rate check, 9 runs, about 2 GPU hours.
   Copy the three `--set training.lr=...` lines it prints.
2. `MODE = "main"` with `ARCH = "resnet50"` and its learning rate: 25 runs, about 6 GPU hours.
3. The same for `ARCH = "efficientnet_b0"`.
4. The same for `ARCH = "vit_s"`.

Each session stays under Kaggle's 12-hour limit. Download `stage3_<name>.zip` after each session.

## Before you run

1. Open **Settings**, set **Accelerator** to a GPU, and set **Internet** to **On**.
2. Attach the dataset `kmader/skin-cancer-mnist-ham10000`.
3. Push the latest code to GitHub first; the notebook clones it.


In [ ]:
import os
import sys

import torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Working directory:", os.getcwd())


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/conorkz/melanoma-xai-bias.git"
REPO_DIR = "/kaggle/working/melanoma-xai-bias"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "timm", "albumentations", "captum", "pyyaml"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
    check=True,
)
print("Repo ready:", os.getcwd())


## Data setup

The next cell links the attached Kaggle HAM10000 dataset if it is present. Otherwise it downloads the original archive from Harvard Dataverse and verifies the MD5 checksums. The download is about 2.8 GB.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)
images_dir = Path("data/HAM10000/images")
metadata_csv = Path("data/raw/HAM10000_metadata.csv")
images_dir.mkdir(parents=True, exist_ok=True)
metadata_csv.parent.mkdir(parents=True, exist_ok=True)

kaggle_root = Path("/kaggle/input/skin-cancer-mnist-ham10000")
linked = False
if (kaggle_root / "HAM10000_metadata.csv").exists():
    shutil.copy2(kaggle_root / "HAM10000_metadata.csv", metadata_csv)
    for part in ("HAM10000_images_part_1", "HAM10000_images_part_2"):
        part_dir = kaggle_root / part
        if part_dir.exists():
            for source in part_dir.glob("*.jpg"):
                target = images_dir / source.name
                if not target.exists():
                    target.symlink_to(source)
    linked = True
    print("Linked HAM10000 from the attached Kaggle dataset")

n_images = len(list(images_dir.glob("*.jpg")))
n_rows = sum(1 for _ in metadata_csv.open()) - 1 if metadata_csv.exists() else 0
if n_images != 10015 or n_rows != 10015:
    if linked:
        for path in images_dir.glob("*.jpg"):
            path.unlink()
    print("Attached dataset incomplete or absent; downloading from Harvard Dataverse")
    subprocess.run([sys.executable, "scripts/download_data.py"], check=True)
    n_images = len(list(images_dir.glob("*.jpg")))
    n_rows = sum(1 for _ in metadata_csv.open()) - 1

print("Metadata rows:", n_rows)
print("Images:", n_images)
assert n_images == 10015 and n_rows == 10015, "HAM10000 is incomplete"
print("Data ready.")


In [ ]:
# Edit these two lines before each session.
MODE = "lr"            # "lr" for the learning-rate check, "main" for an architecture
ARCH = "resnet50"      # resnet50, efficientnet_b0, or vit_s (used when MODE == "main")
LR = None              # learning rate from scripts/select_lr.py, e.g. 0.0001


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)
if MODE == "lr":
    config = "configs/sweep_stage3_lr.yaml"
    extra = []
    SESSION = "lr"
elif MODE == "main":
    if ARCH not in {"resnet50", "efficientnet_b0", "vit_s"}:
        raise ValueError(f"Unknown ARCH {ARCH!r}")
    if LR is None:
        raise ValueError("Set LR from the learning-rate check before a main run")
    config = f"configs/sweep_stage3_{ARCH}.yaml"
    extra = ["--set", f"training.lr={LR}"]
    SESSION = ARCH
else:
    raise ValueError(f"Unknown MODE {MODE!r}")

subprocess.run(
    [sys.executable, "scripts/run_sweep.py", "--config", config, "--skip-existing", "--keep-going", *extra],
    check=True,
)
if MODE == "lr":
    subprocess.run([sys.executable, "scripts/select_lr.py", "--prefix", "stage3lr"], check=True)


In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
prefix = "stage3lr_" if MODE == "lr" else f"stage3_{ARCH}_"
runs = sorted(str(path) for path in Path("outputs/runs").glob(prefix + "*"))
archive = f"stage3_{SESSION}.zip"
subprocess.run(["zip", "-r", "-q", archive, *runs], check=True)
print(f"{len(runs)} runs packaged. Download from the Output pane:")
print(Path(archive).resolve())
